# Validación de Graduados USTA en **CvLAC** — evidencia académica + grupos USTA

**Objetivo.** Verificar cuáles de los graduados (`salidas/graduados_integrado.csv`) tienen
**hoja de vida en CvLAC** (ScienTI / Minciencias), validando con **información académica**
(más segura que el solo nombre) y reforzando con la **pertenencia a grupos de
investigación de la Universidad Santo Tomás** (vía GrupLAC).

## ⚠️ CvLAC no expone la cédula → validamos con evidencia, no con el nombre a secas

CvLAC se identifica por `cod_rh` interno; el dato personal es el **nombre**. Un cruce por
nombre solo es ambiguo (homónimos). Para que sea **seguro**, exigimos evidencia objetiva
de vínculo con la USTA, que puede venir de **dos señales independientes**:

1. **USTA en la formación académica** del perfil CvLAC — regex `santo\s*tomas` sobre
   `formacion_academica` (estudió en la USTA).
2. **Integrante de un grupo de investigación USTA** — el perfil (`cod_rh`) aparece como
   integrante de algún grupo cuya institución avaladora es la USTA, según **GrupLAC**.

> **Regla de validación.** Una coincidencia es **VÁLIDA** si el nombre coincide **y** se
> cumple **al menos una** de las dos señales USTA. La pertenencia a grupo se conoce por
> `cod_rh` (exacta, no por nombre), lo que la hace especialmente fiable.

## Score (0–100)

| Componente | Regla | Puntos |
|------------|-------|--------|
| **Vínculo USTA** (formación **o** grupo) | `flag_usta_form` ó `en_grupo_usta` | **70** (requisito) |
| Integrante de grupo USTA | señal directa por `cod_rh` | +10 |
| **Programa coincide** | Jaccard de *tokens* programa graduado vs CvLAC | +20·(solape) |

**Confianza:** `Alta ≥ 85` · `Media 70–84`. Sin vínculo USTA, la coincidencia no se valida.

## Fuentes (proyecto `Doctorado Matemáticas/scraper_scienti`)

- `data/cvlac/`: hojas de vida CvLAC (JSON) — barrido por `cod_rh` **+ descarga dirigida
  de los integrantes de grupos USTA** (este informe amplió la cobertura USTA).
- `data/gruplac/`: grupos GrupLAC (JSON) con institución avaladora e integrantes (`cod_rh`).


## 1. Configuración e importaciones

In [ ]:
import json, glob, re, time, unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

SALIDA = Path("salidas"); SALIDA.mkdir(exist_ok=True)

SCRAPER = Path(r"c:\Users\cizai\Dropbox\Documentos\Doctorado Matemáticas\scraper_scienti")
CVLAC_DIR = SCRAPER / "data" / "cvlac"
GRUPLAC_DIR = SCRAPER / "data" / "gruplac"
INDICE_PLUS = SALIDA / "cvlac_indice_plus.csv"      # índice enriquecido (caché)
ROSTER_USTA = SALIDA / "cvlac_grupos_usta.csv"      # integrantes de grupos USTA
CVLAC_URL = ("https://scienti.minciencias.gov.co/cvlac/visualizador/"
             "generarCurriculoCv.do?cod_rh={cod_rh}")
print("CvLAC dir :", CVLAC_DIR, "| existe:", CVLAC_DIR.exists())
print("GrupLAC   :", GRUPLAC_DIR, "| existe:", GRUPLAC_DIR.exists())
print("Índice +  :", INDICE_PLUS, "| existe:", INDICE_PLUS.exists())


## 2. Normalización y *tokens* (regex)

In [ ]:
RE_USTA = re.compile(r"santo\s*tomas|u\.?\s*s\.?\s*t\.?a\b|\busta\b")
TOK = re.compile(r"[a-z]+")
STOP = {"de", "del", "la", "el", "en", "y", "para", "con", "las", "los", "e", "una", "un"}

def na(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = unicodedata.normalize("NFKD", s)
    return "".join(c for c in s if not unicodedata.combining(c)).lower()

def name_key(s: str) -> str:
    s = na(s).upper().replace(",", " ")
    s = re.sub(r"[^A-Z\s]", " ", s)
    return " ".join(sorted(t for t in s.split() if len(t) > 1))

def prog_tokens(s: str) -> set:
    return {t for t in TOK.findall(na(s)) if len(t) > 2 and t not in STOP}

def jaccard(a: set, b: set) -> float:
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)

assert name_key("RODRÍGUEZ MONTOYA, MÓNICA") == name_key("Mónica Rodríguez Montoya")
print("OK helpers.")


## 3. Roster de integrantes de grupos USTA (GrupLAC)

Se recorren los GrupLAC cuya **institución avaladora** es la USTA y se reúnen los `cod_rh`
de sus integrantes. Ese conjunto define la señal `en_grupo_usta` (vínculo USTA directo).

> **Adquisición reproducible.** El paso de raspado dirigido ---filtrar los 259 grupos USTA entre los GrupLAC locales, reunir sus 4.546 integrantes y **descargar las hojas de vida CvLAC faltantes** vía el scraper de ScienTI--- está consolidado en el script `_scrape_cvlac_usta.py` de esta carpeta. Aquí se reconstruye el roster desde caché (`cvlac_grupos_usta.csv`).

In [ ]:
def construir_roster_usta(carpeta: Path) -> pd.DataFrame:
    filas = {}
    for f in glob.glob(str(carpeta / "*.json")):
        try:
            d = json.load(open(f, encoding="utf-8"))
        except Exception:
            continue
        insts = " || ".join(d.get("instituciones", []) or [])
        if not RE_USTA.search(na(insts)):
            continue
        for m in d.get("integrantes", []) or []:
            cr = m.get("cod_rh")
            if not cr:
                continue
            cr = str(cr).zfill(10)
            r = filas.setdefault(cr, {"cod_rh": cr, "nombre": m.get("nombre", ""),
                                      "name_key": name_key(m.get("nombre", "")), "n_grupos": 0})
            r["n_grupos"] += 1
    return pd.DataFrame(filas.values())

if ROSTER_USTA.exists():
    roster = pd.read_csv(ROSTER_USTA, dtype={"cod_rh": "string"})
    print(f"Roster cargado de caché: {len(roster):,} integrantes USTA")
else:
    roster = construir_roster_usta(GRUPLAC_DIR)
    roster.to_csv(ROSTER_USTA, index=False, encoding="utf-8-sig")
    print(f"Roster construido: {len(roster):,} integrantes USTA")

cod_grupo_usta = set(roster["cod_rh"].astype(str).str.zfill(10))
print("cod_rh únicos en grupos USTA:", len(cod_grupo_usta))
roster.head(3)


## 4. Índice CvLAC enriquecido (con señales USTA)

Si existe la caché se carga; si no, se reconstruye desde los JSON, extrayendo de cada
perfil el texto de `formacion_academica` y marcando con regex la USTA en formación y en
experiencia. Sobre el índice se añade `en_grupo_usta` (cruce por `cod_rh` con el roster).


In [ ]:
def construir_indice_plus(carpeta: Path) -> pd.DataFrame:
    filas = []
    archivos = glob.glob(str(carpeta / "*.json"))
    print(f"Leyendo {len(archivos):,} perfiles CvLAC...")
    t0 = time.time()
    for i, f in enumerate(archivos, 1):
        try:
            d = json.load(open(f, encoding="utf-8"))
        except Exception:
            continue
        nom = d.get("nombre", "")
        if not nom:
            continue
        fa = d.get("formacion_academica", []) or []
        inst_text = " || ".join(na(e.get("institucion", "")) for e in fa if e.get("institucion"))
        prog_text = " || ".join(na(e.get("programa", "")) for e in fa if e.get("programa"))
        exp_text = na(" ".join(d.get("experiencia", []) or []))
        filas.append({
            "cod_rh": str(d.get("cod_rh", "")).zfill(10),
            "nombre_cvlac": nom,
            "name_key": name_key(nom),
            "nivel_maximo": d.get("nivel_maximo", ""),
            "categoria_minciencias": d.get("categoria_minciencias", ""),
            "total_productos": d.get("total_productos", 0),
            "prog_text": prog_text,
            "flag_usta_form": int(bool(RE_USTA.search(inst_text + " || " + prog_text))),
            "flag_usta_exp": int(bool(RE_USTA.search(exp_text))),
        })
        if i % 10000 == 0:
            print(f"  {i:,}/{len(archivos):,} ({time.time()-t0:.0f}s)")
    return pd.DataFrame(filas)

if INDICE_PLUS.exists():
    cv = pd.read_csv(INDICE_PLUS, dtype={"cod_rh": "string", "name_key": "string"},
                     keep_default_na=False)
    print(f"Índice cargado de caché: {len(cv):,} perfiles")
else:
    cv = construir_indice_plus(CVLAC_DIR)
    cv.to_csv(INDICE_PLUS, index=False, encoding="utf-8-sig")
    print(f"Índice construido y guardado: {len(cv):,} perfiles")

cv["cod_rh"] = cv["cod_rh"].astype("string").str.zfill(10)
for col in ["total_productos", "flag_usta_form", "flag_usta_exp"]:
    cv[col] = pd.to_numeric(cv[col], errors="coerce").fillna(0).astype(int)
cv["en_grupo_usta"] = cv["cod_rh"].isin(cod_grupo_usta).astype(int)
cv["vinculo_usta"] = ((cv["flag_usta_form"] == 1) | (cv["en_grupo_usta"] == 1)).astype(int)
cv["prog_tok"] = cv["prog_text"].map(prog_tokens)
cv["n_homonimos"] = cv.groupby("name_key")["cod_rh"].transform("nunique")
print(f"Perfiles con USTA en formación : {int(cv['flag_usta_form'].sum()):,}")
print(f"Perfiles en grupo USTA         : {int(cv['en_grupo_usta'].sum()):,}")
print(f"Perfiles con vínculo USTA (any): {int(cv['vinculo_usta'].sum()):,}")
cv.head(3)


## 5. Generación de candidatos y *score*

Se cruzan graduados y CvLAC por `name_key` (trae todos los homónimos) y se puntúa cada par
con la evidencia USTA + el solape de programa.


In [ ]:
grad = pd.read_csv(SALIDA / "graduados_integrado.csv", dtype=str, keep_default_na=False)
grad["grad_id"] = np.arange(len(grad))
grad["name_key"] = grad["nombre_completo"].map(name_key)
grad["prog_tok"] = grad["programa"].map(prog_tokens)

cand = grad.merge(cv, on="name_key", how="inner", suffixes=("_g", "_cv"))
print(f"Pares candidato (graduado×perfil): {len(cand):,} | graduados con candidato: {cand['grad_id'].nunique():,}")

def calcular_score(r) -> float:
    if r["vinculo_usta"] != 1:
        return 0.0                                  # sin vínculo USTA -> no validado
    s = 70.0
    if r["en_grupo_usta"] == 1:
        s += 10                                     # integrante de grupo USTA (señal fuerte)
    s += 20 * jaccard(r["prog_tok_g"], r["prog_tok_cv"])
    return float(max(0, min(100, s)))

cand["prog_overlap"] = [jaccard(a, b) for a, b in zip(cand["prog_tok_g"], cand["prog_tok_cv"])]
cand["score"] = cand.apply(calcular_score, axis=1).round(1)

mejor = (cand.sort_values(["score", "total_productos"], ascending=False)
         .drop_duplicates("grad_id").copy())
def nivel(s):
    if s >= 85: return "Alta"
    if s >= 70: return "Media"
    return "No confirmado"
mejor["confianza"] = mejor["score"].map(nivel)
print("\nNivel de confianza:")
print(mejor["confianza"].value_counts().reindex(["Alta", "Media", "No confirmado"]).to_string())
print(f"\nVALIDADOS por vínculo USTA: {int((mejor['score'] >= 70).sum()):,}")
print(f"  por formación USTA       : {int(((mejor['score']>=70) & (mejor['flag_usta_form']==1)).sum()):,}")
print(f"  por grupo USTA           : {int(((mejor['score']>=70) & (mejor['en_grupo_usta']==1)).sum()):,}")


## 6. Consolidación al dataset de graduados

In [ ]:
cols_cv = ["grad_id", "cod_rh", "nombre_cvlac", "nivel_maximo", "categoria_minciencias",
           "total_productos", "flag_usta_form", "flag_usta_exp", "en_grupo_usta",
           "n_homonimos", "prog_overlap", "score", "confianza"]
cruce = grad.merge(mejor[cols_cv], on="grad_id", how="left")
cruce["candidato_nombre"] = cruce["cod_rh"].notna()
cruce["validado_cvlac"]   = cruce["score"] >= 70
cruce["url_cvlac"] = cruce["cod_rh"].map(
    lambda c: CVLAC_URL.format(cod_rh=c) if isinstance(c, str) and c else "")

n = len(cruce)
print(f"Registros de graduados                     : {n:,}")
print(f"  candidatos por nombre (sin confirmar)    : {int(cruce['candidato_nombre'].sum()):,}")
print(f"  VALIDADOS por vínculo USTA               : {int(cruce['validado_cvlac'].sum()):,}")
ced_val = cruce.loc[cruce['validado_cvlac'], 'identificacion'].replace('', np.nan).dropna().nunique()
print(f"  cédulas únicas validadas                 : {ced_val:,}")


## 7. Análisis

### 7.1 Distribución de score y confianza


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
mejor.loc[mejor["score"] >= 70, "score"].plot(kind="hist", bins=15, ax=axes[0],
                                              color="#274690", edgecolor="white")
axes[0].set_title("Score de coincidencias validadas"); axes[0].set_xlabel("Score")
conf = mejor["confianza"].value_counts().reindex(["Alta", "Media", "No confirmado"]).fillna(0)
conf.plot(kind="bar", ax=axes[1], color=["#1B998B", "#E9C46A", "#C0C0C0"])
axes[1].set_title("Coincidencias por nivel de confianza"); axes[1].tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()


### 7.2 Señal de validación (formación USTA vs grupo USTA)

In [ ]:
v = cruce[cruce["validado_cvlac"]]
solo_form = int(((v["flag_usta_form"] == 1) & (v["en_grupo_usta"] == 0)).sum())
solo_grup = int(((v["flag_usta_form"] == 0) & (v["en_grupo_usta"] == 1)).sum())
ambas     = int(((v["flag_usta_form"] == 1) & (v["en_grupo_usta"] == 1)).sum())
serie = pd.Series({"Solo formación USTA": solo_form, "Solo grupo USTA": solo_grup, "Ambas señales": ambas})
display(serie.to_frame("registros validados"))
ax = serie.plot(kind="bar", color=["#274690", "#E07A5F", "#1B998B"])
ax.set_title("Origen de la validación USTA"); ax.tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()


### 7.3 Coincidencias validadas por fuente y por programa

In [ ]:
conf_df = cruce[cruce["validado_cvlac"]]
por_fuente = conf_df.groupby("fuente").size().rename("validados").to_frame()
por_fuente["registros"] = cruce.groupby("fuente").size()
por_fuente["pct"] = (por_fuente["validados"] / por_fuente["registros"] * 100).round(2)
display(por_fuente)

top_prog = conf_df.groupby("programa_norm").size().sort_values(ascending=False).head(15)
ax = top_prog.sort_values().plot(kind="barh", color="#274690")
ax.set_title("Top 15 programas por graduados validados en CvLAC")
ax.set_xlabel("Coincidencias validadas"); ax.set_ylabel("")
plt.tight_layout(); plt.show()


### 7.4 Perfil académico (nivel y categoría Minciencias)

In [ ]:
val = conf_df.drop_duplicates("cod_rh")
print("Nivel máximo (CvLAC):"); display(val["nivel_maximo"].replace("", "(sin dato)").value_counts())
cats = val["categoria_minciencias"].replace("", "(sin categoría)").value_counts()
print("Categoría Minciencias:"); display(cats)
print("Con categoría Minciencias reconocida:",
      int(cats.drop(labels=["(sin categoría)"], errors="ignore").sum()))


### 7.5 Top de graduados-investigadores

In [ ]:
top = (conf_df.sort_values(["total_productos", "score"], ascending=False)
       .drop_duplicates("cod_rh")
       [["nombre_completo", "nombre_cvlac", "fuente", "programa", "nivel_maximo",
         "categoria_minciencias", "total_productos", "flag_usta_form", "en_grupo_usta",
         "prog_overlap", "score", "confianza", "url_cvlac"]]
       .head(20))
display(top)


## 8. Exportación

In [ ]:
ruta_cruce = SALIDA / "graduados_cvlac_cruce.csv"
ruta_match = SALIDA / "graduados_en_cvlac.csv"
cruce.to_csv(ruta_cruce, index=False, encoding="utf-8-sig")

export = (cruce[cruce["validado_cvlac"]]
          .sort_values("score", ascending=False)
          [["identificacion", "nombre_completo", "fuente", "sede", "programa",
            "cod_rh", "nombre_cvlac", "nivel_maximo", "categoria_minciencias",
            "total_productos", "flag_usta_form", "en_grupo_usta", "prog_overlap",
            "n_homonimos", "score", "confianza", "url_cvlac"]])
export.to_csv(ruta_match, index=False, encoding="utf-8-sig")
print("Exportado:")
print(" -", ruta_cruce.resolve())
print(" -", ruta_match.resolve(), f"({len(export):,} validados)")


## 8.bis Personas **USTA en CvLAC** que NO están en la base de graduados

Además de validar a los graduados, identificamos a **todas** las personas con vínculo USTA
en CvLAC (estudiaron en la USTA y/o son integrantes de un grupo de investigación USTA),
**aunque no figuren en nuestras bases de graduados**. Son investigadores de la comunidad
USTA (docentes, estudiantes de posgrado, egresados no incluidos en las bases, etc.).

El vínculo a la base de graduados se evalúa por **nombre** (`name_key`); quienes no tienen
ningún graduado con ese nombre se reportan aquí como *USTA en CvLAC, fuera de graduados*.


In [ ]:
grad_keys = set(grad["name_key"])
usta = cv[cv["vinculo_usta"] == 1].copy()
usta["en_base_graduados"] = usta["name_key"].isin(grad_keys)

n_usta = len(usta)
n_en_base = int(usta["en_base_graduados"].sum())
n_fuera = n_usta - n_en_base
print(f"Perfiles CvLAC con vínculo USTA (total)      : {n_usta:,}")
print(f"  con nombre presente en base de graduados   : {n_en_base:,}")
print(f"  USTA en CvLAC FUERA de la base de graduados : {n_fuera:,}")
print(f"     - solo por grupo USTA                    : {int(((~usta['en_base_graduados']) & (usta['en_grupo_usta']==1) & (usta['flag_usta_form']==0)).sum()):,}")
print(f"     - solo por formación USTA                : {int(((~usta['en_base_graduados']) & (usta['en_grupo_usta']==0) & (usta['flag_usta_form']==1)).sum()):,}")

usta_no_grad = (usta[~usta["en_base_graduados"]]
                .sort_values("total_productos", ascending=False)
                .copy())
usta_no_grad["url_cvlac"] = usta_no_grad["cod_rh"].map(lambda c: CVLAC_URL.format(cod_rh=c))
cols = ["cod_rh", "nombre_cvlac", "nivel_maximo", "categoria_minciencias",
        "total_productos", "flag_usta_form", "en_grupo_usta", "url_cvlac"]
ruta = SALIDA / "cvlac_usta_no_graduados.csv"
usta_no_grad[cols].to_csv(ruta, index=False, encoding="utf-8-sig")
print("\nExportado:", ruta.resolve(), f"({len(usta_no_grad):,} perfiles USTA fuera de graduados)")
display(usta_no_grad[cols].head(10))


## 8.ter Categorización de las revistas (OpenAlex)

Para caracterizar la calidad y el alcance de los artículos de los investigadores
validados, se extrae el **ISSN** de cada artículo de su CvLAC y se consulta la revista en
**OpenAlex** (catálogo bibliográfico abierto), obteniendo país, acceso abierto (DOAJ),
indexación y campo de conocimiento. Patrón **cargar-o-extraer** (usa los CSV en `salidas/`
si existen). El mismo ISSN permitiría, como extensión, un cruce con **Publindex**.

In [ ]:
import json, re, time, requests
# 1) Extraer ISSN de los artículos de los validados (cargar-o-extraer)
f_issn = SALIDA / "cvlac_issn.csv"
if f_issn.exists():
    issn_df = pd.read_csv(f_issn, dtype=str)
    print("ISSN cargados de caché:", len(issn_df))
else:
    val = pd.read_csv(SALIDA / "graduados_en_cvlac.csv", dtype=str)
    cods = val.drop_duplicates("cod_rh")["cod_rh"].dropna().tolist()
    RE_ISSN = re.compile(r"ISSN:\s*([0-9]{4}-?[0-9]{3}[0-9Xx])")
    filas = []
    for c in cods:
        p = CVLAC_DIR / f"{str(c).zfill(10)}.json"
        if not p.exists(): continue
        d = json.load(open(p, encoding="utf-8"))
        for it in d.get("produccion", {}).get("articulos", {}).get("items", []) or []:
            for m in RE_ISSN.finditer(str(it)):
                v = m.group(1).upper().replace("-", "")
                filas.append({"cod_rh": c, "issn": v[:4] + "-" + v[4:] if len(v) == 8 else v})
    issn_df = pd.DataFrame(filas)
    issn_df.to_csv(f_issn, index=False, encoding="utf-8-sig")
    print("ISSN extraídos:", len(issn_df))
print("Menciones de revista:", len(issn_df), "| ISSN únicos:", issn_df["issn"].nunique())

In [ ]:
# 2) Consultar cada ISSN en OpenAlex (cargar-o-extraer)
f_oa = SALIDA / "cvlac_issn_openalex.csv"
if f_oa.exists():
    oa = pd.read_csv(f_oa, dtype=str)
    print("OpenAlex cargado de caché:", len(oa))
else:
    SES = requests.Session(); MAIL = "cizaineam@gmail.com"; out = []
    issns = sorted(issn_df["issn"].dropna().unique())
    for i, issn in enumerate(issns, 1):
        rec = {"issn": issn, "found": 0, "name": "", "country": "", "doaj": "", "oa": "", "field": "", "works": 0}
        try:
            r = SES.get(f"https://api.openalex.org/sources/issn:{issn}", params={"mailto": MAIL}, timeout=30)
            if r.status_code == 200:
                d = r.json(); rec["found"] = 1; rec["name"] = d.get("display_name") or ""
                rec["country"] = d.get("country_code") or ""; rec["doaj"] = int(bool(d.get("is_in_doaj")))
                rec["oa"] = int(bool(d.get("is_oa"))); rec["works"] = d.get("works_count") or 0
                tp = d.get("topics") or []
                if tp: rec["field"] = tp[0].get("field", {}).get("display_name") or ""
        except Exception: pass
        out.append(rec); time.sleep(0.07)
    oa = pd.DataFrame(out); oa.to_csv(f_oa, index=False, encoding="utf-8-sig")
    print("OpenAlex consultado:", len(oa))
for c in ["found", "doaj", "oa"]:
    oa[c] = pd.to_numeric(oa[c], errors="coerce").fillna(0).astype(int)

In [ ]:
# 3) Categorización a nivel de artículo
m = issn_df.merge(oa, on="issn", how="left")
m["found"] = m["found"].fillna(0).astype(int)
N = len(m); ind = m[m["found"] == 1]
print(f"Artículos con ISSN: {N:,}")
print(f"  en revista indexada en OpenAlex: {int(m['found'].sum()):,} ({m['found'].mean()*100:.1f}%)")
print(f"  en revista colombiana          : {int((ind['country']=='CO').sum()):,} ({(ind['country']=='CO').mean()*100:.1f}%)")
print(f"  en revista de acceso abierto    : {int(ind['doaj'].sum()):,} ({ind['doaj'].mean()*100:.1f}%)")

campo = ind[ind["field"].fillna("") != ""]["field"].value_counts().head(10)
ax = campo.sort_values().plot(kind="barh", color="#5C415D", figsize=(7.5,4.2))
ax.set_title("Artículos por campo de la revista (OpenAlex)"); ax.set_xlabel("Artículos")
plt.tight_layout(); plt.show()
print("\nTop países de edición:"); display(ind["country"].replace("","(s/d)").value_counts().head(6).to_frame("artículos"))

## 9. Conclusiones y limitaciones

**Resultado.** La validación se apoya en **evidencia objetiva de vínculo con la USTA**, no
en el nombre solo: (1) USTA en la formación académica del perfil CvLAC, y/o (2)
pertenencia a un grupo de investigación USTA (GrupLAC, por `cod_rh`). Esto reduce
fuertemente los falsos positivos por homónimo. La cobertura se amplió descargando los
CvLAC de los integrantes de grupos USTA que faltaban.

**Limitaciones**
- **No es cruce por cédula** (CvLAC no la expone). El vínculo graduado→CvLAC sigue siendo
  por **nombre**; la evidencia USTA confirma la identidad, pero conviene abrir `url_cvlac`
  para decisiones individuales.
- **Universo parcial de CvLAC.** Aunque se reforzó con los grupos USTA, el barrido general
  por `cod_rh` no cubre todo CvLAC: pueden faltar graduados con perfil pero sin vínculo
  USTA detectable ni pertenencia a grupo USTA (**falsos negativos**).
- **GrupLAC** refleja integrantes reportados; un egresado puede tener CvLAC sin estar en un
  grupo USTA ni declarar la USTA en su formación, y entonces no se valida aquí.

> Junto con SECOP (proveedor del Estado) y RUES (emprendimiento), este cruce aporta la
> faceta **investigativa** del impacto del egresado, validada con evidencia académica.
